# Signatures comparison — TRUE validation cohort (CHESS-1336)

Companion to the published Figure 4 notebook (`Signatures comparison.ipynb`) and to
`Signatures comparison NEW.ipynb`. The new sorted-cell cohort delivered with
[Jira OD-128](https://bostongene.atlassian.net/browse/OD-128) is used here as a
**true held-out validation cohort** — this is the CHESS-1336 rerun of the original
cell-type FGES benchmark on validation data.

**Scope:** the 16 in-scope FGES (`MAP_RAW`). The four rare-GOI FGES
(`Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`,
`Main4_Plasma_cells`, i.e. `EXCLUDED_FGES_RARE`) are deferred to the separate
rare-types notebook (75/25 stratified holdouts, `signature_validation.benchmark.splits`).

**Baseline:** the published Figure-4 scores (`data/mapping_ssgseas.pkl`) are re-scored
through the *same* `plot_sens_spec_scatter` code path, so the validation-vs-baseline
sensitivity/specificity deltas use an identical metric definition on both cohorts.

**Random-FGES baseline:** v1 random gene lists are reused verbatim from
`data/msigdb_gmt.pkl` and only re-scored on the validation cohort, so ranks stay
comparable.

**Outputs:** everything is written under `../plots/` and `../tables/` with a
`_validation` suffix; v1 and NEW outputs are never overwritten.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import s3fs
import seaborn as sns
import zarr
from loguru import logger

from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    MAP_RAW,
    build_mapping,
    intersect_controls_with_cohort,
    load_new_cohort_annotation,
)
from signature_validation.benchmark.plotting import (
    _scatter_source_for_signature,
    plot_sens_spec_scatter,
    plot_signature_heatmap,
    plot_violin_per_source,
)
from signature_validation.benchmark.scoring import (
    compute_mapping_ssgseas,
    compute_out_table,
    fdr_correct_out,
)
from signature_validation.benchmark.signatures import (
    count_random_fges,
    harmonize_gmt_to_index,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.plotting.plotting import cells_p
from signature_validation.utils.fges_utils import get_metric_for_signature

sns.set_style("white")
plt.rcParams["svg.fonttype"] = "none"

In [3]:
import bioreactor

## §3 — Load validation annotation

> The loader does **not** filter by QC (its docstring assumes a pre-filtered file), so we apply the OD-128 filter explicitly: `Technical_QC == True` AND `Decision_deconvolution_without_parent != False` (30418 -> ~17322 samples).

In [4]:
# --- Validation-cohort inputs (local copies, committed under data/) ---
VALIDATION_ANNOT_PATH = Path(
    "/home/jovyan/projects/Signature_Validation/Paper_Code_and_Figures/Figure_4/Cell_type_FGES_comparison/data/sorted_cells_to_check_all_annot.tsv"
)

# Expressions live on S3 as a zarr v3 array (obs x var), NOT per-GSE TSV.
# read_expressions does NOT apply here; §4 reads this zarr directly (see load_osrp_expressions).
OSRP_ZARR = (
    "bostongene-eurynome-exchange/raw_data/v2/expressions/osrp_tier1_expressions.zarr"
)

# --- Baseline / signature inputs (committed in the repo) ---
V1_GMT_PICKLE = Path("../data/msigdb_gmt.pkl")              # real local pickle
BASELINE_SSGSEAS_PATH = Path("../data/mapping_ssgseas.pkl")  # real local pickle (199 MB)

# --- Outputs (repo folders, `_validation` suffix; v1/NEW never overwritten) ---
OUTPUT_DIR = Path("../plots")
TABLES_DIR = Path("../tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_SSGSEAS_PATH = OUTPUT_DIR / "mapping_ssgseas_validation.pkl"
OUT_TSV_PATH = TABLES_DIR / "out_validation.tsv"
CMP_TSV_PATH = TABLES_DIR / "comparison_validation_vs_baseline.tsv"
HEATMAP_PATH = OUTPUT_DIR / "signature_heatmap_validation.svg"

# Fail loudly, early, if a local input is missing.
for label, p in (
    ("validation annotation", VALIDATION_ANNOT_PATH),
    ("v1 GMT pickle", V1_GMT_PICKLE),
    ("baseline ssgseas pickle", BASELINE_SSGSEAS_PATH),
):
    if not p.exists():
        logger.error("{} not found: {}", label, p)

logger.info("outputs -> {} and {}", OUTPUT_DIR, TABLES_DIR)

2026-07-28 18:23:27.006 | INFO     | __main__:<module>:36 - outputs -> ../plots and ../tables


In [5]:
validation_annot = load_new_cohort_annotation(VALIDATION_ANNOT_PATH)

# Explicit QC filter (not applied inside load_new_cohort_annotation).
# NOTE: Technical_QC is bool, but Decision_deconvolution_without_parent is object
# ('True'/'False'/NaN strings). Comparing a string to the python bool `False` is
# ALWAYS True (str != bool), so `!= False` would silently keep 'False' rows.
# Compare against the STRING "False"; NaN (-> 'nan') correctly passes (Kassandra
# made no prediction, not a rejection).
qc_mask = (validation_annot["Technical_QC"] == True) & (
    validation_annot["Decision_deconvolution_without_parent"].astype(str) != "False"
)
validation_annot = validation_annot[qc_mask]
logger.info("annotation after QC filter: {} samples", len(validation_annot))
validation_annot["Cell_type"].value_counts()

/home/jovyan/projects/Signature_Validation/src/signature_validation/utils/utils.py:541: DtypeWarning: Columns (0: Cell_type_details, 1: To_check, 2: QC_good, 3: QC_score, 4: Decision, 5: Expression_comment, 6: Spike_in, 7: Response, 8: Prior_surgery, 9: Diagnosis_comment, 10: Measurement_method, 11: Deconvolution_cell_type, 12: Unnamed: 12, 13: OS_flag, 14: OS_duration, 15: Prior_medication, 16: Severity, 17: Zero_expressions) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(
2026-07-28 18:23:27.382 | INFO     | signature_validation.benchmark.cohorts:load_new_cohort_annotation:224 - loaded 30418 samples across 217 cell types from /home/jovyan/projects/Signature_Validation/Paper_Code_and_Figures/Figure_4/Cell_type_FGES_comparison/data/sorted_cells_to_check_all_annot.tsv
2026-07-28 18:23:27.405 | INFO     | __main__:<module>:13 - annotation after QC filter: 17322 samples


Cell_type
CD4_T_cells                          1720
CD8_T_cells                          1371
Monocytes                            1155
Epithelium                           1143
Macrophages                           957
                                     ... 
Trophoblast_cells                       1
Stromal_Cells                           1
Mature_neutrophils                      1
Orthochromatophilic_erythroblasts       1
Mononuclear_cells                       1
Name: count, Length: 199, dtype: int64

## §4 — Load validation expressions (from osrp zarr)

> Expressions are a zarr v3 array (obs x var: SRX samples x gene symbols), not the per-GSE TSVs `read_expressions` expects. We read the needed rows straight from S3 (lazy chunks) and return a genes x samples frame. Only samples whose id is in the zarr `obs_names` are returned (BGP internal samples are absent from open-source osrp).

In [6]:
def load_osrp_expressions(
    sample_ids: list[str],
    zarr_path: str = OSRP_ZARR,
    log2: bool = True,
) -> pd.DataFrame:
    """Load osrp zarr expressions for the given samples as a genes x samples frame.

    Parameters
    ----------
    sample_ids : list[str]
        Sample ids from the annotation index; the intersection with ``obs_names``
        is taken automatically.
    zarr_path : str
        Bucket+key of the zarr array (no ``s3://`` scheme).
    log2 : bool
        Apply ``log2(TPM + 1)`` (v1-pipeline transform). Set False if the array is
        already in log space (check ``expr.max()``: ~15-20 => raw TPM; ~4-5 => log).

    Returns
    -------
    pd.DataFrame
        Genes x samples (index = ``var_names``, columns = matched ``obs_names``).

    Raises
    ------
    KeyError
        If no ``sample_ids`` are present in the zarr ``obs_names``.
    """
    fs = s3fs.S3FileSystem()
    store = zarr.storage.FsspecStore(fs, path=zarr_path)  # zarr v3
    z = zarr.open(store, mode="r")

    obs = list(z.attrs["obs_names"])
    var = list(z.attrs["var_names"])
    pos = {s: i for i, s in enumerate(obs)}

    want = [s for s in sample_ids if s in pos]
    if not want:
        raise KeyError("no sample_ids found in osrp zarr obs_names")
    rows = sorted(pos[s] for s in want)
    logger.info("osrp zarr: {} / {} samples matched", len(rows), len(sample_ids))

    x = z.oindex[rows, :]  # (n_want, n_genes) float32
    expr = pd.DataFrame(x.T, index=var, columns=[obs[r] for r in rows])
    if log2:
        expr = np.log2(expr + 1)
    logger.info("expressions: {} genes x {} samples", expr.shape[0], expr.shape[1])
    return expr


validation_expr = load_osrp_expressions(list(validation_annot.index))
validation_expr.shape

/tmp/ipykernel_6173/2606290144.py:30: ZarrUserWarning: fs (<s3fs.core.S3FileSystem object at 0x7f934014fe30>) was not created with `asynchronous=True`, this may lead to surprising behavior
  store = zarr.storage.FsspecStore(fs, path=zarr_path)  # zarr v3
/home/jovyan/projects/Signature_Validation/.venv/lib/python3.12/site-packages/zarr/storage/_fsspec.py:249: ZarrUserWarning: fs (<s3fs.core.S3FileSystem object at 0x7f934014fe30>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return type(self)(
2026-07-28 18:23:27.799 | INFO     | __main__:load_osrp_expressions:41 - osrp zarr: 10790 / 17322 samples matched
2026-07-28 18:23:50.434 | INFO     | __main__:load_osrp_expressions:47 - expressions: 20062 genes x 10790 samples


(20062, 10790)

In [7]:
import sys, os, zarr
print("executable:", sys.executable)
print("PYTHONPATH env:", os.environ.get("PYTHONPATH"))
print("sys.path:", sys.path)
print("zarr file:", zarr.__file__)
print("zarr version:", zarr.__version__)


executable: /home/jovyan/projects/Signature_Validation/.venv/bin/python
PYTHONPATH env: None
sys.path: ['/opt/conda/lib/python312.zip', '/opt/conda/lib/python3.12', '/opt/conda/lib/python3.12/lib-dynload', '', '/home/jovyan/projects/Signature_Validation/.venv/lib/python3.12/site-packages', '/home/jovyan/projects/Signature_Validation/src']
zarr file: /home/jovyan/projects/Signature_Validation/.venv/lib/python3.12/site-packages/zarr/__init__.py
zarr version: 3.2.1


## §5 — Build FGES mapping (16 in-scope FGES, scoped to the validation cohort)

In [8]:
mapping = build_mapping(annotation=validation_annot)
controls_present = intersect_controls_with_cohort(CONTROLS_ORDER, validation_annot)
logger.info(
    "in-scope FGES: {}; controls present in validation cohort: {}",
    len(mapping),
    len(controls_present),
)
for sign, bucket in mapping.items():
    logger.info(
        "{}: GOI={}, Control={}, Deleted={}",
        sign,
        bucket["Goi"],
        len(bucket["Control"]),
        len(bucket["Deleted_controls"]),
    )

2026-07-28 18:23:50.493 | INFO     | __main__:<module>:3 - in-scope FGES: 15; controls present in validation cohort: 33
2026-07-28 18:23:50.493 | INFO     | __main__:<module>:9 - Main4_Th1_signature: GOI=['Th1_cells'], Control=30, Deleted=4
2026-07-28 18:23:50.493 | INFO     | __main__:<module>:9 - Main4_CD8_T_cells: GOI=['CD8_T_cells'], Control=31, Deleted=3
2026-07-28 18:23:50.494 | INFO     | __main__:<module>:9 - Main4_Treg: GOI=['Tregs'], Control=31, Deleted=3
2026-07-28 18:23:50.494 | INFO     | __main__:<module>:9 - Main4_Neutrophil_signature: GOI=['Neutrophils'], Control=33, Deleted=1
2026-07-28 18:23:50.494 | INFO     | __main__:<module>:9 - Main4_Mast_cell_signature: GOI=['Mast_cells'], Control=33, Deleted=1
2026-07-28 18:23:50.495 | INFO     | __main__:<module>:9 - Main4_Effector_cells: GOI=['CD8_T_cells', 'NK_cells'], Control=32, Deleted=1
2026-07-28 18:23:50.495 | INFO     | __main__:<module>:9 - Main4_Follicular_helper_T_cells: GOI=['Follicular_T_helpers'], Control=30, De

## §6 — Reuse v1 gene lists (byte-identical) and harmonize to the validation index

## §7 — Compute ssGSEA on the validation cohort

In [9]:
import bioreactor


In [10]:
v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
in_scope_fges = [k for k in MAP_RAW if k not in EXCLUDED_FGES_RARE]
v1_gmt = select_msigdb_gmt_subset(v1_gmt_full, in_scope_fges)

for sign in in_scope_fges:
    assert sign in v1_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    n_random = count_random_fges(v1_gmt[sign])
    assert n_random == 10, f"{sign}: expected 10 RANDOM_FGES, got {n_random}"

msigdb_gmt = harmonize_gmt_to_index(v1_gmt, validation_expr.index)
logger.info(
    "msigdb_gmt: {} FGES, {} sub-signatures total",
    len(msigdb_gmt),
    sum(len(v) for v in msigdb_gmt.values()),
)

2026-07-28 18:23:50.608 | INFO     | signature_validation.benchmark.signatures:load_v1_msigdb_gmt:58 - v1 GMT loaded: 19 FGES, 2867 sub-signatures total
/home/jovyan/projects/Signature_Validation/src/signature_validation/ssgsea_calc/ssgsea_calc.py:132: UserWarning: 2 hits for gene SEPTIN6
  warnings.warn("{} hits for gene {}".format(len(hits), cg))
/home/jovyan/projects/Signature_Validation/src/signature_validation/ssgsea_calc/ssgsea_calc.py:132: UserWarning: 2 hits for gene GATD3
  warnings.warn("{} hits for gene {}".format(len(hits), cg))
/home/jovyan/projects/Signature_Validation/src/signature_validation/ssgsea_calc/ssgsea_calc.py:132: UserWarning: 2 hits for gene TENT4A
  warnings.warn("{} hits for gene {}".format(len(hits), cg))
/home/jovyan/projects/Signature_Validation/src/signature_validation/ssgsea_calc/ssgsea_calc.py:132: UserWarning: 2 hits for gene EPRS1
  warnings.warn("{} hits for gene {}".format(len(hits), cg))
/home/jovyan/projects/Signature_Validation/src/signature_val

In [ ]:
mapping_ssgseas = compute_mapping_ssgseas(
    public_cells_expr=validation_expr,
    public_cells_annot=validation_annot,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
)

for sign in mapping_ssgseas:
    if sign in EXCLUDED_FGES_RARE:
        continue
    assert mapping_ssgseas[sign]["Goi"], f"{sign}: GOI cohort is empty"
    for ct, frame in mapping_ssgseas[sign]["Goi"].items():
        logger.info("{} GOI {}: {} samples", sign, ct, frame.shape[0])

with open(MAPPING_SSGSEAS_PATH, "wb") as fh:
    pickle.dump(mapping_ssgseas, fh, pickle.HIGHEST_PROTOCOL)
logger.info("wrote {}", MAPPING_SSGSEAS_PATH)

ssGSEA per FGES:   0%|          | 0/15 [00:00<?, ?it/s]2026-07-28 18:32:24.527 | WARNING  | signature_validation.benchmark.scoring:compute_mapping_ssgseas:62 - Main4_Th1_signature/Control/Plasma_B_cells: no samples with expressions; skipping
2026-07-28 18:32:24.529 | WARNING  | signature_validation.benchmark.scoring:compute_mapping_ssgseas:62 - Main4_Th1_signature/Control/Plasmablasts: no samples with expressions; skipping
ssGSEA per FGES:   7%|▋         | 1/15 [00:41<09:46, 41.89s/it]2026-07-28 18:34:26.184 | WARNING  | signature_validation.benchmark.scoring:compute_mapping_ssgseas:62 - Main4_CD8_T_cells/Control/Plasma_B_cells: no samples with expressions; skipping
2026-07-28 18:34:26.186 | WARNING  | signature_validation.benchmark.scoring:compute_mapping_ssgseas:62 - Main4_CD8_T_cells/Control/Plasmablasts: no samples with expressions; skipping


## §8 — Per-signature × cell-type stats table (+ deterministic FDR)

In [ ]:
out = compute_out_table(mapping_ssgseas, mapping, msigdb_gmt, controls_present)
out = fdr_correct_out(out, controls_present)
out.to_csv(OUT_TSV_PATH, sep="\t")
logger.info("wrote {} ({} rows x {} cols)", OUT_TSV_PATH, *out.shape)
out.head()

> **Checkpoint.** §1–§8 above form the minimal working skeleton and must run top-to-bottom with no errors before §8b–§10.

## §8b — Weighted F-score benchmark (Figure 4E–G)

> Central quantitative result of the paper: per-sub-signature **weighted F1** with the ROC threshold **closest to (0,1)** (`get_metric_for_signature`), **10 bootstrap rounds** for class balance, sampling size per Methods (`50` / `n_GOI` / `n_controls`), and **CV** across rounds. GOI = positive class, all controls = negative.

In [ ]:
N_FSCORE_ROUNDS = 10
FSCORE_SEED = 42
FSCORE_TSV_PATH = TABLES_DIR / "fscore_benchmark_validation.tsv"


def compute_fscore_benchmark(
    mapping_ssgseas: dict,
    n_rounds: int = N_FSCORE_ROUNDS,
    seed: int = FSCORE_SEED,
) -> pd.DataFrame:
    """Weighted-F1 benchmark per sub-signature (Methods: cell-type FGES comparison).

    For each FGES and each sub-signature (source), ssGSEA scores in the GOI are the
    positive class and scores across all control cell types are the negative class.
    ``n_rounds`` bootstrap rounds balance the classes; the sampling size follows the
    paper: ``50`` if ``n_GOI < 50``, else ``n_GOI`` if ``n_GOI < n_controls``, else
    ``n_controls``. Per round the weighted F1 is computed by
    :func:`get_metric_for_signature` (ROC threshold closest to (0,1)).

    Parameters
    ----------
    mapping_ssgseas : dict
        Output of :func:`compute_mapping_ssgseas` (``{FGES: {Goi|Control: {ct: df}}}``).
    n_rounds : int
        Number of bootstrap rounds (Methods: 10).
    seed : int
        RNG seed for reproducibility.

    Returns
    -------
    pd.DataFrame
        One row per sub-signature; columns ``signature, fges, source, F1_mean, CV,
        n_goi, n_ctrl``.
    """
    rng = np.random.default_rng(seed)
    records: list[dict] = []
    for fges, groups in mapping_ssgseas.items():
        if not groups["Goi"] or not groups["Control"]:
            continue
        goi_df = pd.concat(groups["Goi"].values())
        ctrl_frames = list(groups["Control"].values())
        for signat in goi_df.columns:
            goi_scores = goi_df[signat].dropna()
            ctrl_scores = pd.concat(
                [c[signat] for c in ctrl_frames if signat in c.columns]
            ).dropna()
            n_goi, n_ctrl = len(goi_scores), len(ctrl_scores)
            if n_goi == 0 or n_ctrl == 0:
                continue
            if n_goi < 50:
                samp = 50
            elif n_goi < n_ctrl:
                samp = n_goi
            else:
                samp = n_ctrl
            f1s: list[float] = []
            for _ in range(n_rounds):
                gp = goi_scores.iloc[rng.integers(0, n_goi, samp)]
                cp = ctrl_scores.iloc[rng.integers(0, n_ctrl, samp)]
                series = pd.concat([gp, cp]).reset_index(drop=True)
                labels = pd.Series([1] * samp + [0] * samp)
                f1s.append(get_metric_for_signature(series, labels)["F1"])
            arr = np.asarray(f1s, dtype=float)
            records.append(
                {
                    "signature": signat,
                    "fges": fges,
                    "source": "BG" if signat == fges else _scatter_source_for_signature(signat),
                    "F1_mean": float(arr.mean()),
                    "CV": float(arr.std() / arr.mean()) if arr.mean() else float("nan"),
                    "n_goi": n_goi,
                    "n_ctrl": n_ctrl,
                }
            )
    return pd.DataFrame(records)


fscore_df = compute_fscore_benchmark(mapping_ssgseas)
fscore_df.to_csv(FSCORE_TSV_PATH, sep="\t", index=False)
logger.info(
    "F-score benchmark: {} sub-signatures across {} sources -> {}",
    len(fscore_df),
    fscore_df["source"].nunique(),
    FSCORE_TSV_PATH,
)
fscore_df.groupby("source")[["F1_mean", "CV"]].mean().sort_values("F1_mean", ascending=False)

## §8c — Internal vs public F-score (Wilcoxon paired + FDR)

> Per Methods, F-scores are compared with a **paired Wilcoxon test** (FDR-corrected). Here: internal `BG` F1 vs each other source, paired across FGES cell types.

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

FSCORE_CMP_PATH = TABLES_DIR / "fscore_comparison_validation.tsv"

bg_f1 = fscore_df[fscore_df["source"] == "BG"].groupby("fges")["F1_mean"].mean()
cmp_rows: list[dict] = []
for source in fscore_df["source"].unique():
    if source == "BG":
        continue
    other = fscore_df[fscore_df["source"] == source].groupby("fges")["F1_mean"].mean()
    common = bg_f1.index.intersection(other.index)
    if len(common) < 3:  # paired test needs enough pairs
        logger.warning("skip {}: only {} paired FGES", source, len(common))
        continue
    stat, p = wilcoxon(bg_f1.loc[common], other.loc[common])
    cmp_rows.append(
        {
            "source": source,
            "n_fges": int(len(common)),
            "BG_mean_F1": float(bg_f1.loc[common].mean()),
            "source_mean_F1": float(other.loc[common].mean()),
            "wilcoxon_p": float(p),
        }
    )

cmp_df = pd.DataFrame(cmp_rows)
if len(cmp_df):
    cmp_df["FDR"] = multipletests(cmp_df["wilcoxon_p"], method="fdr_bh")[1]
    cmp_df = cmp_df.sort_values("source_mean_F1", ascending=False)
cmp_df.to_csv(FSCORE_CMP_PATH, sep="\t", index=False)
logger.info("wrote F-score comparison -> {}", FSCORE_CMP_PATH)
cmp_df

## §9 — Validation vs published baseline (sensitivity / specificity)

Option A: the published Figure-4 scores (`data/mapping_ssgseas.pkl`) are re-scored
through the same `plot_sens_spec_scatter` code path used for validation, so the delta
uses an identical metric definition on both cohorts. Run `git lfs pull` first — the
baseline pickle is LFS-tracked.

In [ ]:
if not BASELINE_SSGSEAS_PATH.exists():
    raise FileNotFoundError(f"baseline scores not found: {BASELINE_SSGSEAS_PATH}")
with open(BASELINE_SSGSEAS_PATH, "rb") as fh:
    head = fh.read(64)
if head.startswith(b"version https://git-lfs"):
    raise RuntimeError(
        f"{BASELINE_SSGSEAS_PATH} is an unresolved git-LFS pointer; run `git lfs pull`"
    )
with open(BASELINE_SSGSEAS_PATH, "rb") as fh:
    baseline_ssgseas = pickle.load(fh)
logger.info("baseline scores loaded: {} FGES", len(baseline_ssgseas))

In [ ]:
# `val_avg` is produced here and reused by the §10 scatter; `base_avg` uses a
# throwaway suffix (its SVGs are not part of the deliverable).
val_avg = plot_sens_spec_scatter(
    mapping_ssgseas=mapping_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
    suffix="_validation",
)
base_avg = plot_sens_spec_scatter(
    mapping_ssgseas=baseline_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
    suffix="_baseline_tmp",
)

# Remove the throwaway baseline SVGs so only `_validation` artefacts remain.
for tmp_svg in OUTPUT_DIR.glob("*_baseline_tmp*"):
    tmp_svg.unlink()

In [ ]:
rows: list[dict] = []
for sign in val_avg:
    for axis in ("Sensitivity", "Specificity"):
        for src, series in val_avg[sign][axis].items():
            baseline_series = base_avg.get(sign, {}).get(axis, {}).get(src)
            rows.append(
                {
                    "FGES": sign,
                    "axis": axis,
                    "source": src,
                    "validation": float(series.mean()),
                    "baseline": (
                        float(baseline_series.mean())
                        if baseline_series is not None
                        else float("nan")
                    ),
                }
            )
cmp = pd.DataFrame(rows)
cmp["delta"] = cmp["validation"] - cmp["baseline"]
cmp.to_csv(CMP_TSV_PATH, sep="\t", index=False)
logger.info("wrote {} ({} rows)", CMP_TSV_PATH, len(cmp))
cmp.head(20)

## §10 — Figures (suffix `_validation`; rare cell types starred)

The plot helpers already carry the `_validation` suffix and star rare cell types /
rare FGES automatically. The validation sens/spec scatter was produced in §9
(`val_avg`); it is not re-run here.

In [ ]:
plot_violin_per_source(mapping_ssgseas, save_dir=OUTPUT_DIR, suffix="_validation")
plot_signature_heatmap(
    mapping_ssgseas=mapping_ssgseas,
    out_df=out,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
    annotation=validation_annot,
    controls_order=controls_present,
    palette={ct: cells_p[ct] for ct in controls_present if ct in cells_p},
    save_path=HEATMAP_PATH,
    short=True,
)
logger.info("plots saved under {}", OUTPUT_DIR)

## §10b — F-score & CV by source (Figure 4F–G)

> Weighted F-score and CV per FGES source on the validation cohort. Red line = random prediction (F=0.5). Higher F1 with lower CV = better, more consistent discrimination.

In [ ]:
FSCORE_FIG_PATH = OUTPUT_DIR / "fscore_by_source_validation.svg"

order = (
    fscore_df.groupby("source")["F1_mean"].median().sort_values(ascending=False).index
)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=fscore_df, x="source", y="F1_mean", order=order, ax=axes[0])
axes[0].axhline(0.5, color="red", ls="--", lw=1, label="Random prediction")
axes[0].set_title("Weighted F-score by source (validation)")
axes[0].set_ylabel("F1 (weighted)")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45)
sns.boxplot(data=fscore_df, x="source", y="CV", order=order, ax=axes[1])
axes[1].set_title("CV by source (validation)")
axes[1].tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FSCORE_FIG_PATH)
logger.info("F-score figure -> {}", FSCORE_FIG_PATH)

## §11 — Rare cell types (out of scope here)

FGES whose GOI is rare in the validation cohort — `Main4_Th17_signature`,
`Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` —
are deferred to a separate rare-types notebook. That notebook reuses the original
cohort, generates 10 stratified 75/25 holdouts via
`signature_validation.benchmark.splits.stratified_holdout_indices` (stratified by
BG-FGES score median × GOI/Control), scores ssGSEA on each test fold and aggregates
with `aggregate_score_over_splits`. Rare cell types are starred on the resulting
figures.